## DSCI 100 Project:
# How do economic and structural influences affect the total cryptocurrency adoption across countries


#### Introduction: 
As traditional financial and economic systems face increasing strain in many parts of the world, cryptocurrency has emerged as a potential alternative financial tool and has been adopted by many different countries. This project examines whether broader forms of economic vulnerability, such as financial exclusion, income levels, macroeconomic instability, and structural differences, including other countries' corruption indices, are associated with higher levels of cryptocurrency adoption across countries.

The data shown below is drawn from the World Bank Corruption Index, Currency Depreciation data from Kaggle and Chainalysis, and covers mainly the period from 2022 to 2025.


<p align="center">
  <img src="photo/Crypto 2.jpg" width="600">
</p>

*Figure 1: Cryptocurrency as an emerging alternative to traditional financial systems.*

## Part 1: Baseline Model
To begin, we explore whether traditional economic indicators can explain why some countries adopt cryptocurrency more than others. We use three key variables — inflation, GDP per capita, and bank access — as predictors. Inflation captures monetary instability, GDP reflects overall income levels, and bank access measures the availability of formal financial services. By fitting a linear regression with these variables, we establish a baseline understanding of the economic factors behind cryptocurrency adoption.


### a) Cleaning the data:
Loading inflation data from the World Bank and calculating the average inflation rate across 2022–2024 for each country.

In [25]:
import pandas as pd
inflation_raw = pd.read_csv("data/Inflation.csv", skiprows=3)
year_cols = ["2022", "2023", "2024"]
inflation = inflation_raw.loc[:, ["Country Name"] + year_cols]
inflation["avg_inflation"] = inflation.loc[:, year_cols].mean(axis=1)
inflation = inflation.loc[:, ["Country Name", "avg_inflation"]]
inflation

,Country Name,avg_inflation
0,Aruba,NaN
1,Africa Eastern and Southern,7.590817
2,Afghanistan,0.822069
3,Africa Western and Central,5.592821
4,Angola,21.079962
...,...,...
261,Kosovo,6.048095
262,"Yemen, Rep.",NaN
263,South Africa,5.825423
264,Zambia,12.287787


Loading the 2024 Global Crypto Adoption Index and converting country rankings into an adoption score scaled from 0 to 1:

In [26]:
import pandas as pd
crypto_raw = pd.read_csv("data/crypto_adoption_2024.csv")
crypto = crypto_raw.rename(columns={"country": "Country Name"})
max_rank = crypto["overall_rank_2024"].max()
crypto["adoption_score"] = (max_rank - crypto["overall_rank_2024"]) / (max_rank - 1)
crypto

,Country Name,region,overall_rank_2024,sub1_centralized_value,sub2_retail_centralized,sub3_defi_value,sub4_retail_defi,adoption_score
0,India,Central & Southern Asia and Oceania,1,1,1,3,2,1.000000
1,Nigeria,Sub-Saharan Africa,2,5,2,2,3,0.993333
2,Indonesia,Central & Southern Asia and Oceania,3,6,6,1,1,0.986667
3,United States,North America,4,2,12,4,4,0.980000
4,Vietnam,Central & Southern Asia and Oceania,5,3,3,6,5,0.973333
...,...,...,...,...,...,...,...,...
146,Mauritius,Sub-Saharan Africa,147,126,130,150,150,0.026667
147,Belize,Latin America,148,152,158,123,149,0.020000
148,Rep. of Congo,Sub-Saharan Africa,149,147,142,151,144,0.013333
149,Mali,Sub-Saharan Africa,150,88,122,156,152,0.006667


Merging inflation data with the crypto adoption dataset by country name, and sorting by adoption rank:

In [27]:
inflation_crypto = inflation.merge(crypto, on="Country Name", how="inner")
inflation_crypto.sort_values(by="overall_rank_2024")

,Country Name,avg_inflation,region,overall_rank_2024,sub1_centralized_value,sub2_retail_centralized,sub3_defi_value,sub4_retail_defi,adoption_score
54,India,5.767071,Central & Southern Asia and Oceania,1,1,1,3,2,1.000000
89,Nigeria,25.582945,Sub-Saharan Africa,2,5,2,2,3,0.993333
53,Indonesia,3.353455,Central & Southern Asia and Oceania,3,6,6,1,1,0.986667
125,United States,5.022888,North America,4,2,12,4,4,0.980000
123,Ukraine,13.178215,Eastern Europe,6,7,5,5,6,0.966667
...,...,...,...,...,...,...,...,...,...
0,Aruba,NaN,Latin America,146,154,157,138,138,0.033333
85,Mauritius,7.143752,Sub-Saharan Africa,147,126,130,150,150,0.026667
18,Belize,4.652098,Latin America,148,152,158,123,149,0.020000
78,Mali,4.962226,Sub-Saharan Africa,150,88,122,156,152,0.006667


Loading bank access data from the World Bank, keeping only the 2024 values and removing missing entries:

In [28]:
bank_raw = pd.read_csv("data/Bank access.csv", skiprows=3)
bank = bank_raw.loc[:, ["Country Name", "2024"]]
bank = bank.dropna()
bank = bank.rename(columns={"2024": "bank_access"})
bank

,Country Name,bank_access
5,Albania,46.069251
9,Argentina,81.744245
10,Armenia,71.373473
13,Australia,98.010378
14,Austria,99.508497
...,...,...
259,World,78.742404
261,Kosovo,64.161772
263,South Africa,81.125966
264,Zambia,72.702425


Loading GDP per capita data and calculating the average GDP across 2022–2024 for each country:

In [29]:
GDP= pd.read_csv("data/GDP.csv", skiprows=4)
GDP["avg_gdp"] = GDP[["2022", "2023", "2024"]].mean(axis=1)
GDP=GDP[["Country Name","2022", "2023", "2024","avg_gdp"]]
GDP

,Country Name,2022,2023,2024,avg_gdp
0,Aruba,30975.998912,35718.753119,39498.594129,35397.782054
1,Africa Eastern and Southern,1679.327622,1571.449189,1615.396356,1622.057722
2,Afghanistan,357.261153,413.757895,NaN,385.509524
3,Africa Western and Central,2138.473153,1841.855064,1411.337029,1797.221749
4,Angola,3682.113151,2916.136633,2665.874448,3088.041411
...,...,...,...,...,...
261,Kosovo,5290.950065,6221.206188,7023.065985,6178.407413
262,"Yemen, Rep.",NaN,NaN,NaN,NaN
263,South Africa,6534.248678,6034.272090,6267.186814,6278.569194
264,Zambia,1447.123101,1330.727806,1187.109434,1321.653447


Selecting the relevant columns from each dataset and merging all four datasets into a single dataframe by country name:

In [30]:
inflation_final = inflation.loc[:, ["Country Name", "avg_inflation"]]
crypto_final = crypto.loc[:, ["Country Name", "adoption_score", "overall_rank_2024"]]
bank_final = bank.loc[:, ["Country Name", "bank_access"]]
GDP_final = GDP.loc[:, ["Country Name", "avg_gdp"]]

merged_all = inflation_final.merge(GDP_final, on="Country Name", how="inner") \
                            .merge(crypto_final, on="Country Name", how="inner") \
                            .merge(bank_final, on="Country Name", how="inner")

merged_all

,Country Name,avg_inflation,avg_gdp,adoption_score,overall_rank_2024,bank_access
0,Albania,4.566474,9621.868950,0.280000,109,46.069251
1,Argentina,141.934541,14064.606545,0.906667,15,81.744245
2,Armenia,3.630281,7762.425245,0.493333,77,71.373473
3,Australia,5.117575,64943.960686,0.746667,39,98.010378
4,Austria,6.432973,55728.385154,0.513333,74,99.508497
...,...,...,...,...,...,...
110,United States,5.022888,80741.183929,0.980000,4,97.023098
111,Uzbekistan,10.343830,2873.111931,0.786667,33,59.658383
112,South Africa,5.825423,6278.569194,0.806667,30,81.125966
113,Zambia,12.287787,1321.653447,0.620000,58,72.702425


In [31]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

baseline_data = merged_all.loc[:, [
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access"
]].dropna()

X = baseline_data.loc[:, ["avg_inflation", "avg_gdp", "bank_access"]]
y = baseline_data["adoption_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

print("Training R²:", model.score(X_train, y_train))
print("Testing R²:", model.score(X_test, y_test))

Training R²: 0.08138068377743701
Testing R²: -0.04274416293297967


### b) Regression Result:
Fitting a linear regression model using inflation, GDP, and bank access as predictors of crypto adoption score

In [32]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

baseline_data = merged_all.loc[:, [
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access"
]].dropna()

X = baseline_data.loc[:, ["avg_inflation", "avg_gdp", "bank_access"]]
y = baseline_data["adoption_score"]

model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X, y)
model.score(X, y)

0.06340298076872386

Visualizing the baseline model performance by plotting actual vs predicted adoption scores:

In [33]:
import pandas as pd
import altair as alt

# Create dataframe with actual and predicted values
baseline_plot_df = pd.DataFrame({
    "actual": y,
    "predicted": model.predict(X)
})

# Scatter plot
points = alt.Chart(baseline_plot_df).mark_circle(opacity=0.5).encode(
    x=alt.X("actual", title="Actual Crypto Adoption Score"),
    y=alt.Y("predicted", title="Predicted Crypto Adoption Score")
).properties(
    title="Baseline Model: Actual vs Predicted Crypto Adoption"
)

points

alt.Chart(...)

Usescross-validation with GridSearchCV to test different values of k in a KNN regression model (with standardized features), computes the RMSE for each, and visualizes how model performance changes with k to select the best one.

In [10]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score

from sklearn.model_selection import GridSearchCV

param_grid = {"kneighborsregressor__n_neighbors": range(1, 21)}

grid = GridSearchCV(
    make_pipeline(StandardScaler(), KNeighborsRegressor()),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error")

grid.fit(X, y)

results_df = pd.DataFrame(grid.cv_results_)[
    ["param_kneighborsregressor__n_neighbors", "mean_test_score"]
].rename(columns={
    "param_kneighborsregressor__n_neighbors": "k"
})

results_df["rmse"] = -results_df["mean_test_score"]
alt.Chart(results_df).mark_line(point=True).encode(
    x=alt.X("k", title="Number of Neighbors (K)"),
    y=alt.Y("rmse", title="Cross-Validated RMSE")
).properties(
    title="KNN Regression: Choosing K"
)

alt.Chart(...)

### Results & Findings: Baseline Model

The baseline model produces an R² of approximately 0.063, indicating that inflation, GDP, and bank access together have limited explanatory power in predicting cryptocurrency adoption. This suggests that traditional economic indicators alone are insufficient, and other structural or institutional factors may play a more important role.

While the model improves slightly when moving from K = 1 to larger values, the overall error remains substantial and relatively flat beyond K ≈ 5. This suggests that increasing model complexity does not meaningfully improve predictive accuracy.

Overall, the consistently high RMSE indicates that these economic variables alone provide limited predictive power for cryptocurrency adoption. This reinforces the idea that additional structural, behavioral, or institutional variables are likely necessary to better explain variation in adoption rates.


## Part 2: Regional Differences in Crypto Adoption
Given the weak relationships observed in the regression analysis, it is necessary to consider alternative perspectives. In particular, examining regional differences may reveal patterns that are not captured by individual economic indicators.


Adding region information to the merged dataset and selecting the relevant columns for regional analysis:

In [34]:
merged_all = merged_all.merge(
    crypto.loc[:, ["Country Name", "region"]], on="Country Name", how="inner")
model_data2 = merged_all.loc[:, [
    "Country Name",
    "region",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "adoption_score"
]].dropna()

model_data2.head()

,Country Name,region,avg_inflation,avg_gdp,bank_access,adoption_score
0,Albania,Central & Western Europe,4.566474,9621.868950,46.069251,0.280000
1,Argentina,Latin America,141.934541,14064.606545,81.744245,0.906667
2,Armenia,Middle East & North Africa,3.630281,7762.425245,71.373473,0.493333
3,Australia,Central & Southern Asia and Oceania,5.117575,64943.960686,98.010378,0.746667
4,Austria,Central & Western Europe,6.432973,55728.385154,99.508497,0.513333


Calculating and visualizing the average crypto adoption score by region:

In [35]:
region_avg = model_data2.groupby("region")["adoption_score"].mean().reset_index()
alt.Chart(region_avg).mark_bar().encode(
    y=alt.Y("region", sort='-x', title="Region"),
    x=alt.X("adoption_score", title="Adoption Score"),
    color="region"
).properties(title="Average Crypto Adoption by Region")

alt.Chart(...)

#### Result & Findings: 
The regional analysis shows that North America has the highest average crypto adoption score, followed by Central & Southern Asia and Oceania and Eastern Europe. Sub-Saharan Africa and Latin America have the lowest adoption levels. This variation suggests that regional and structural conditions play an important role in shaping cryptocurrency adoption beyond individual economic indicators.


## Part 3: Similarity among High-Adoption Countries
While regional differences highlight geographic patterns in adoption, it is also worth examining whether high-adoption countries share common economic characteristics. By comparing the economic profiles of top adopters with the overall sample, we can assess whether a distinct set of conditions drives high adoption.


Identifying high-adoption countries as those in the top 25% of adoption scores:

In [36]:
h4_data = merged_all.loc[:, [
    "Country Name",
    "region",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "adoption_score"
]].dropna()

top_25 = h4_data["adoption_score"].quantile(0.75)
high_adoption = h4_data.loc[h4_data["adoption_score"] >= top_25]

high_adoption.head()

,Country Name,region,avg_inflation,avg_gdp,bank_access,adoption_score
1,Argentina,Latin America,141.934541,14064.606545,81.744245,0.906667
9,Bangladesh,Central & Southern Asia and Oceania,9.348735,2620.306594,43.283973,0.773333
15,Brazil,Latin America,6.080378,9989.823824,86.381178,0.940000
17,Canada,North America,4.354462,54939.158984,98.404713,0.886667
20,China,Eastern Asia,0.808847,13074.977345,89.383032,0.873333


Calculating the average inflation, GDP, and bank access for high-adoption countries:

In [37]:
high_adoption[["avg_inflation", "avg_gdp", "bank_access"]].mean()

avg_inflation       12.484310
avg_gdp          18682.601698
bank_access         75.406054
dtype: float64

Calculating the same averages for all countries as a comparison:

In [38]:
h4_data[["avg_inflation", "avg_gdp", "bank_access"]].mean()

avg_inflation        9.195744
avg_gdp          20291.901503
bank_access         72.445462
dtype: float64

Checking which regions are most represented among high-adoption countries:

In [39]:
high_adoption["region"].value_counts()

Central & Southern Asia and Oceania    8
Central & Western Europe               5
Sub-Saharan Africa                     5
Latin America                          4
North America                          2
Eastern Asia                           2
Eastern Europe                         2
Middle East & North Africa             1
Name: region, dtype: int64

Comparing the economic profiles of high-adoption countries versus all countries using bar charts:

In [40]:
import pandas as pd
import altair as alt

high_inflation = high_adoption["avg_inflation"].mean()
high_gdp = high_adoption["avg_gdp"].mean()
high_bank = high_adoption["bank_access"].mean()

all_inflation = h4_data["avg_inflation"].mean()
all_gdp = h4_data["avg_gdp"].mean()
all_bank = h4_data["bank_access"].mean()


compare_df = pd.DataFrame({
    "group": ["High Adoption", "All Countries"],
    "avg_inflation": [high_inflation, all_inflation],
    "avg_gdp": [high_gdp, all_gdp],
    "bank_access": [high_bank, all_bank]
})


chart1 = alt.Chart(compare_df).mark_bar().encode(
    y=alt.Y("group", title="Group"),
    x=alt.X("avg_inflation", title="Average Inflation"),
    color="group"
).properties(title="Inflation")


chart2 = alt.Chart(compare_df).mark_bar().encode(
    y=alt.Y("group", title="Group"),
    x=alt.X("avg_gdp", title="Average GDP"),
    color="group"
).properties(title="GDP")


chart3 = alt.Chart(compare_df).mark_bar().encode(
    y=alt.Y("group", title="Group"),
    x=alt.X("bank_access", title="Bank Access"),
    color="group"
).properties(title="Bank Access")

chart1 & chart2 & chart3

alt.VConcatChart(...)

#### Result/Findings:
High-adoption countries show slightly higher average inflation and bank access compared to all countries, while their average GDP is marginally lower. However, the differences across all three variables are relatively small, suggesting that high-adoption countries do not share a single distinct economic profile. This indicates that traditional economic indicators alone cannot identify what makes a country a high adopter, motivating the inclusion of additional variables in the next section.


## Part 4: Extended Model – Currency Instability and Institutional Factors
To address the limitations of the baseline model, this section introduces additional variables that better reflect real-world drivers of cryptocurrency adoption. In particular, currency depreciation and corruption are included to capture monetary instability and institutional trust.

Currency depreciation reflects the loss of value in local currencies, which may encourage individuals to adopt cryptocurrencies as alternative stores of value. The corruption index is used as a proxy for institutional trust, where lower trust in government and financial systems may increase reliance on decentralized financial tools.

Loading currency depreciation and corruption index data for the extended model.

In [41]:
import pandas as pd
currency_clean = pd.read_csv("data/currency_clean.csv")
corruption_index=pd.read_csv("data/index.csv")
corruption_index

,CPI Rank,Country,Country Code,Region,Corruption Perceptions Index (CPI),Standard Error,Lower Confidence Interval,Upper Confidence Interval,World Bank CPIA,World Economic Forum EOS,...,African Development Bank CPIA,IMD World Competitiveness Yearbook,Bertelsmann Foundation Sustainable Governance Index,World Justice Project Rule of Law Index,PRS International Country Risk Guide,Varities of Democracy Project,Economist Intelligence Unit Country Ratings,Freedom House Nations in Transit Ratings,PERC Asia Risk Guide,Sources
0,1,New Zealand,NZL,Asia Pacific,90,2.56,86,94,NaN,90.0,...,NaN,95.0,99.0,79.0,93.0,NaN,90.0,NaN,NaN,7
1,1,Denmark,DNK,Europe and Central Asia,90,2.46,86,94,NaN,85.0,...,NaN,98.0,99.0,85.0,93.0,NaN,90.0,NaN,NaN,7
2,3,Finland,FIN,Europe and Central Asia,89,1.46,87,92,NaN,91.0,...,NaN,94.0,90.0,85.0,93.0,NaN,90.0,NaN,NaN,7
3,4,Sweden,SWE,Europe and Central Asia,88,1.33,85,90,NaN,86.0,...,NaN,86.0,90.0,85.0,93.0,NaN,90.0,NaN,NaN,7
4,5,Switzerland,CHE,Europe and Central Asia,86,1.57,83,89,NaN,80.0,...,NaN,88.0,90.0,NaN,85.0,NaN,90.0,NaN,NaN,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,170,Sudan,SDN,Sub-Saharan Africa,14,2.99,9,19,2.0,NaN,...,11.0,NaN,NaN,NaN,6.0,22.0,19.0,NaN,NaN,7
172,173,Syria,SYR,Middle East and North Africa,13,1.97,10,16,NaN,NaN,...,NaN,NaN,NaN,NaN,15.0,12.0,19.0,NaN,NaN,5
173,174,Korea (North),PRK,Asia Pacific,12,1.39,10,15,NaN,NaN,...,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,3
174,175,South Sudan,SSD,Sub-Saharan Africa,11,3.21,5,16,2.0,NaN,...,5.0,NaN,NaN,NaN,NaN,19.0,NaN,NaN,NaN,5


Merging currency depreciation data into the main dataset:

In [42]:
extended_data = merged_all.merge(
    currency_clean,
    on="Country Name",
    how="left")
extended_data

,Country Name,avg_inflation,avg_gdp,adoption_score,overall_rank_2024,bank_access,region,currency_dep
0,Albania,4.566474,9621.868950,0.280000,109,46.069251,Central & Western Europe,-0.192556
1,Argentina,141.934541,14064.606545,0.906667,15,81.744245,Latin America,260.593585
2,Armenia,3.630281,7762.425245,0.493333,77,71.373473,Middle East & North Africa,NaN
3,Australia,5.117575,64943.960686,0.746667,39,98.010378,Central & Southern Asia and Oceania,0.596872
4,Austria,6.432973,55728.385154,0.513333,74,99.508497,Central & Western Europe,NaN
...,...,...,...,...,...,...,...,...
110,United States,5.022888,80741.183929,0.980000,4,97.023098,North America,NaN
111,Uzbekistan,10.343830,2873.111931,0.786667,33,59.658383,Central & Southern Asia and Oceania,NaN
112,South Africa,5.825423,6278.569194,0.806667,30,81.125966,Sub-Saharan Africa,1.108909
113,Zambia,12.287787,1321.653447,0.620000,58,72.702425,Sub-Saharan Africa,NaN


Renaming columns in the corruption dataset and merging both currency and corruption data into the main dataset:

In [43]:
corruption = corruption_index.rename(columns={
    "Country": "Country Name",
    "Corruption Perceptions Index (CPI)": "corruption_index"
})

extended_data = merged_all.merge(
    currency_clean,
    on="Country Name",
    how="left"
).merge(corruption,on="Country Name",how="left")


Fitting the extended regression model with inflation, GDP, bank access, currency depreciation, and corruption index as predictors:

In [44]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

model_data_ext = extended_data.loc[:, [
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]].dropna()

X_ext = model_data_ext.loc[:, [
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]]
y_ext = model_data_ext["adoption_score"]

model_ext = make_pipeline(StandardScaler(), LinearRegression())
model_ext.fit(X_ext, y_ext)

model_ext.score(X_ext, y_ext)

0.1939708226821042

Visualizing the relationship between corruption index and crypto adoption:

In [45]:
import pandas as pd
import altair as alt


model_data_ext = extended_data.loc[:, [
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]].dropna()


model_data_ext["predicted"] = model_ext.predict(
    model_data_ext.loc[:, [
        "avg_inflation",
        "avg_gdp",
        "bank_access",
        "currency_dep",
        "corruption_index"
    ]]
)

plot_data = model_data_ext.loc[model_data_ext["currency_dep"] < 50]

alt.Chart(plot_data).mark_circle().encode(
    x=alt.X("currency_dep", title="Currency Depreciation"),
    y=alt.Y("adoption_score", title="Crypto Adoption Score")
).properties(
    title="Crypto Adoption vs Currency Depreciation"
)

alt.Chart(...)

Re-fitting the extended model to confirm results:

In [48]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

model_data_ext = extended_data.loc[:, [
    "adoption_score",
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]].dropna()

X_ext = model_data_ext.loc[:, [
    "avg_inflation",
    "avg_gdp",
    "bank_access",
    "currency_dep",
    "corruption_index"
]]
y_ext = model_data_ext["adoption_score"]

model_ext = make_pipeline(StandardScaler(), LinearRegression())
model_ext.fit(X_ext, y_ext)

model_ext.score(X_ext, y_ext)

0.1939708226821042

Plotting actual vs predicted adoption scores for the extended model:

In [49]:
import pandas as pd
import altair as alt

# Predict using actual data
model_data_ext["predicted"] = model_ext.predict(
    model_data_ext.loc[:, [
        "avg_inflation",
        "avg_gdp",
        "bank_access",
        "currency_dep",
        "corruption_index"
    ]]
)

# Scatter plot
plot2 = alt.Chart(model_data_ext).mark_circle().encode(
    x=alt.X("corruption_index", title="CPI Score (Higher = Less Corruption)"),
    y=alt.Y("adoption_score", title="Crypto Adoption Score")
).properties(
    title="Predicted Crypto Adoption by CPI Score"
)
plot2

alt.Chart(...)

#### Results/Findings
The extended model improves R² from 0.063 to approximately 0.19, showing that currency depreciation and corruption index add meaningful explanatory power. Currency depreciation has a negative relationship with adoption, suggesting that extreme instability may limit participation rather than encourage it. 

The corruption index is also negatively related since higher CPI scores indicate less corruption, this means countries with more corruption tend to have higher adoption, likely reflecting lower trust in formal institutions. Overall, cryptocurrency adoption is shaped by a combination of economic pressure and institutional context, not just traditional economic indicators.



## Conclusion

This project finds that traditional economic indicators alone have limited power in explaining cryptocurrency adoption. Regional differences and institutional factors such as corruption and currency instability play a more meaningful role. The extended model nearly tripled the R² from 0.063 to 0.19, confirming that cryptocurrency adoption is shaped by a combination of economic pressure and institutional context rather than any single factor.


<p align="center">
  <img src="photo/crypto_crash.jpeg" width="600">
</p>

<p align="center"><i>Figure 2: Cryptocurrency remains unstable in the financial market.</i></p>